# Compound Emotion Recognition - MobileNet V2 + QAT



In [ ]:
# RUN THIS FIRST THEN REFRESH THE KERNAL
# Make sure the line is comment before run all

#!pip install -q tensorflow-model-optimization

# Import Library

In [ ]:
# ============================================================
# CE-OWN COMPOUND EMOTION - MobileNetV2 + QAT
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import csv
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import joblib
from PIL import Image
from pathlib import Path
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from scipy.optimize import differential_evolution
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score
)

print("TensorFlow:", tf.__version__)
print("TFMOT:", tfmot.__version__)

# Dataset

In [ ]:
# ============================================================
# DATASET PATH
# ============================================================

DATASET_ROOT = "/kaggle/input/datasets/qystyy/sorted-compound-emotion/CE_split_SC_rmv"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")
VAL_DIR  = os.path.join(DATASET_ROOT, "valid")

IMG_SIZE = (160, 160)
BATCH_SIZE = 16
 
SEED = 42

print("Train:", TRAIN_DIR)
print("Test :", TEST_DIR)
print("Validation :", VAL_DIR)

## Image Preprocessing 

Some images in the dataset is in a different format; especially .tiff

This is because .tiff image format cant be run thorugh the model training epoch, which will cause overfitting/ underfitting as the data dont have the suppose data

In [ ]:
# Copy dataset to working directory
source = "/kaggle/input/datasets/qystyy/sorted-compound-emotion/CE_split_SC_rmv"
dest = "/kaggle/working/CE_split_SC_rmv"

print("Copying dataset to working directory (this may take a minute)...")
if not os.path.exists(dest):
    shutil.copytree(source, dest)
    print("✅ Copy complete!")
else:
    print("Dataset already in working directory")

# Now update your paths
DATASET_ROOT = "/kaggle/working/CE_split_SC_rmv"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")
VAL_DIR   = os.path.join(DATASET_ROOT, "valid")

print(f"\nUsing dataset from: {DATASET_ROOT}")

# Now you can convert TIFF to PNG
def convert_tiff_to_png(root_dir):
    """Convert all TIFF files to PNG"""
    converted = 0
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith('.tiff'):
                tiff_path = os.path.join(root, file)
                png_path = tiff_path.replace('.tiff', '.png').replace('.TIFF', '.png')
                
                try:
                    img = Image.open(tiff_path)
                    img.convert('RGB').save(png_path, 'PNG')
                    os.remove(tiff_path)
                    converted += 1
                    
                    if converted % 20 == 0:
                        print(f"Converted: {converted}")
                        
                except Exception as e:
                    print(f"Error: {e}")
    
    return converted

# Convert all datasets
for folder in ["train", "test", "valid"]:
    folder_path = os.path.join(DATASET_ROOT, folder)
    count = convert_tiff_to_png(folder_path)
    print(f"{folder.upper()}: Converted {count} TIFF files to PNG")

In [ ]:
# Sanity checks with the format after the conversion

def check_file_formats(root_dir):
    """Check what formats exist"""
    formats = {}
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            if ext in ['.png', '.jpg', '.jpeg', '.tiff', '.gif', '.bmp']:
                formats[ext] = formats.get(ext, 0) + 1
    
    return formats

# Check all datasets
for folder in ["train", "test", "valid"]:
    folder_path = os.path.join(DATASET_ROOT, folder)
    formats = check_file_formats(folder_path)
    
    print(f"\n{folder.upper()}:")
    for fmt, count in sorted(formats.items()):
        print(f"  {fmt}: {count}")

## Load dataset from the working Directory

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("\nClasses:")
for i, name in enumerate(class_names):
    print(f"{i:2d}: {name}")

print("\nNumber of classes:", NUM_CLASSES)

## Data Preprocessing

In [ ]:
# ============================================================
# DATA PREPROCESSING
# ============================================================

def preprocess_images(images, labels):
    # Convert to float32
    images = tf.cast(images, tf.float32)

    # Convert grayscale images to RGB
    # If image is already RGB, keep it unchanged
    if images.shape[-1] == 1:
        images = tf.image.grayscale_to_rgb(images)

    # Resize to target size
    images = tf.image.resize(images, IMG_SIZE)

    # Normalize pixel values from [0, 255] → [0, 1]
    images = images / 255.0

    return images, labels

train_ds = train_ds.map(
    preprocess_images,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds = test_ds.map(
    preprocess_images,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    preprocess_images,
    num_parallel_calls=tf.data.AUTOTUNE
)

# ============================================================
# OPTIMIZE DATASETS
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

for images, labels in train_ds.take(1):
    print("Shape:", images.shape)
    print("Data type:", images.dtype)
    print("Min:", tf.reduce_min(images).numpy())
    print("Max:", tf.reduce_max(images).numpy())

## Class Distribution + Class Weight Calculation

In [ ]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================

def get_class_counts(dataset, class_names):
    counts = np.zeros(len(class_names), dtype=int)

    for _, labels in dataset:
        labels = labels.numpy()

        for label in labels:
            counts[label] += 1

    return counts


train_counts = get_class_counts(train_ds, class_names)
test_counts = get_class_counts(test_ds, class_names)
val_counts = get_class_counts(val_ds, class_names)

distribution_df = pd.DataFrame({
    "Emotion": class_names,
    "Train": train_counts,
    "Test": test_counts,
    "Validation": val_counts,
    "Total": train_counts + test_counts + val_counts
})

print(distribution_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    distribution_df["Emotion"],
    distribution_df["Total"]
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of Images")
plt.title("RAF-CE Compound Emotion Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPUTE CLASS WEIGHTS
# ============================================================

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=np.concatenate([
        labels.numpy()
        for _, labels in train_ds
    ])
)

class_weights = {
    i: float(weight)
    for i, weight in enumerate(class_weights_array)
}

print("Class weights:")
for i, name in enumerate(class_names):
    print(f"{i:2d} | {name:25s} | {class_weights[i]:.4f}")

## Data Augmentation

In [ ]:
# ============================================================
# DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),  # Reduced for frozen backbone
    tf.keras.layers.RandomZoom(0.08),      # Reduced
    # Removed: Brightness, Contrast, Translation - too aggressive
], name="data_augmentation")



# Base Model Training - FER Trained MobileNet-V2

In [ ]:
# ============================================================
# BUILD MOBILENETV2 AS BASE MODEL
# ============================================================
"""
# 1. Build base model
#      This base model is to generate a keras file for a pretrained FER model to be used as the base model
#      for the compound emotion training. With a pretrained model, the model has an early insight in the base
#      7 emotion for them to directly pinpoint in the compound emotions.
#      If there's no existing FER model, train the model with a REF-13 dataset or RAF-DB dataset to get the model.

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet', 
    #alpha = 0.35,
)
"""
# Load pretrained model
# make sure the model file is in KERAS format
pretrained_model = tf.keras.models.load_model(
    "/kaggle/input/models/qystyy/raf-db-model-trained/keras/default/1/rafce_mobilenetv2_best.keras"
)

# Extract JUST the backbone
mobilenet_backbone = pretrained_model.get_layer("mobilenetv2_1.00_160")
mobilenet_backbone.trainable = False

inputs = tf.keras.Input(
    shape=IMG_SIZE + (3,),
    name="image"
)

x = data_augmentation(inputs)

x = mobilenet_backbone(x)

# NOTE: The commented layers are the layers used in the BASE MODEL training with FER dataset
#       If needed to the base model train, uncomment those section

#x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)  # (None, 1280)
x = tf.keras.layers.GlobalAveragePooling2D()(mobilenet_backbone.output)  # (None, 1280)
#x = tf.keras.layers.Dense(512, activation='relu')(x)                                          # REDUCE THE AMOUNT OF HEAD CLASSIFICATION LAYER
#x = tf.keras.layers.Dropout(0.4)(x)                                                           # This is mainly for CE as to increase the model performance
x = tf.keras.layers.Dense(128, activation='relu')(x)   # change DENSE, 128
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation='softmax',
    dtype='float32',
    name='predictions',
)(x)

#model = tf.keras.Model(base_model.input, outputs=output, name='MobileNetV2_QAT')             # FOR BASE MODEL TRAINING
model = tf.keras.Model(mobilenet_backbone.input, outputs=output, name='MobileNetV2_QAT')

# 2. Save the clean base model BEFORE quantizing
base_save_path = os.path.join('base_model.keras')
model.save(base_save_path)
print('Base model saved.')

print(f'Total layers : {len(model.layers)}')
print(f'Total params : {model.count_params():,}')
model.summary()

## QAT Input

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5.  QAT WRAPPER  (RUN ONCE ONLY)
# ══════════════════════════════════════════════════════════════════════
# QAT wrapper will only be applied once to the model. The wrapper will be carried through all phases of training, evaluation, and inference.
# Only COMPOUND EMOTIONS MODEL TRAINING will run through with QAT wrapper.
# Please comment this section if you want to train the BASE MODEL TRAINING with FER dataset. The base model training will not run through with QAT wrapper.

print('Inserting fake-quantisation nodes ...')
model = tfmot.quantization.keras.quantize_model(model)
print(f'QAT model — layers after wrapping: {len(model.layers)}')
print('Fake-quant nodes inserted. QAT wrapper will be carried through all phases.')

## Callbacks

In [ ]:
# ============================================================
# CALLBACKS
# ============================================================

checkpoint_path = "rafce_mobilenetv2_best.keras"

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        mode = 'min',
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        min_delta = 0.0005,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

## Stage 1: Model Training [Freeze 143 Layers]

In [ ]:
# ============================================================
# STAGE 1 COMPILE
# ============================================================
"""
# This commented section is only for base model training 
# For base model training, only run Stage 1 and Stage 2
# Stage 3 is reserved only for CE Model training

# Freeze everything except the final 10 layers
for layer in base_model.layers[:-10]:
    layer.trainable = False

# Keep BatchNorm frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(
    1 for layer in model.layers
    if layer.trainable
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5   # dropping LR for QAT attempt
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",]
)"""

# Precision Metric class as Sparse Categorical Cross-entropy has no precision metric
class SparsePrecision(tf.keras.metrics.Precision):
    def __init__(self, name='sparse_precision', **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        # Convert class probabilities to predicted class integers
        y_pred_classes = tf.math.argmax(y_pred, axis=-1)
        
        # Cast to match the data type of y_true (usually int32 or int64)
        y_pred_classes = tf.cast(y_pred_classes, y_true.dtype)
        
        # Super expects both inputs to have the same shape/type
        return super().update_state(y_true, y_pred_classes, sample_weight)
    

mobilenet_backbone.trainable = True

# Freeze everything except the final 15 layers
# Start with the last 15 layers to unfreeze as completely frozen layers will overfit

for layer in mobilenet_backbone.layers[:-15]:
    layer.trainable = False

# Keep BatchNorm frozen
for layer in mobilenet_backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(
    1 for layer in model.layers
    if layer.trainable
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        #learning_rate=5e-5
        learning_rate=1e-5   # dropping LR for QAT attempt
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        SparsePrecision()]
)

In [ ]:
# ============================================================
# STAGE 1 TRAINING
# CLASSIFIER ONLY
# ============================================================

EPOCHS_STAGE1 = 15

history_stage1 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks,
    class_weight=class_weights
)

## Stage 2: Fine-tuning [Freeze 128 layers]

In [ ]:
# ============================================================
# STAGE 2 - FINE TUNING
# ============================================================
"""
base_model.trainable = True

# Freeze everything except the final 40 layers
for layer in base_model.layers[:-40]:
    layer.trainable = False

# Keep BatchNorm frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(
    1 for layer in model.layers
    if layer.trainable
)

print("Outer model layers:", len(model.layers))
print("MobileNetV2 layers:", len(base_model.layers))

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=4e-6  
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

"""
mobilenet_backbone.trainable = True

# Freeze everything except the final 30 layers
for layer in mobilenet_backbone.layers[:-30]:
    layer.trainable = False

# Keep BatchNorm frozen
for layer in mobilenet_backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(
    1 for layer in model.layers
    if layer.trainable
)

print("Outer model layers:", len(model.layers))
print("MobileNetV2 layers:", len(mobilenet_backbone.layers))

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=5e-6
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        SparsePrecision()]
)

In [ ]:
# ============================================================
# STAGE 2 TRAINING
# ============================================================

EPOCHS_STAGE2 = 10

history_stage2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks,
    class_weight=class_weights
)

## Stage 3: Fine Tuning [Freeze 98 Layers]

In [ ]:
# ============================================================
# STAGE 3 - FINE TUNING
# ============================================================

mobilenet_backbone.trainable = True

# Freeze everything except the final 60 layers
for layer in mobilenet_backbone.layers[:-60]:
    layer.trainable = False

# Keep BatchNorm frozen
for layer in mobilenet_backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(
    1 for layer in model.layers
    if layer.trainable
)

print("Outer model layers:", len(model.layers))
print("MobileNetV2 layers:", len(mobilenet_backbone.layers))

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=3e-6
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        SparsePrecision()]
)

In [ ]:
# ============================================================
# STAGE 3 TRAINING
# ============================================================
"""
Comment this section when running the BASE MODEL TRAINING
"""
EPOCHS_STAGE3 = 5

history_stage3 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_STAGE3,
    callbacks=callbacks,
    class_weight=class_weights
)

# Result Evaluation 

## Loss & Accuracy Learning Curve

In [ ]:
# ============================================================
# COMBINE HISTORIES
# ============================================================
"""
# To print out the history for the model training of FER BASE MODEL

def combine_histories(h1, h2):
    history = {}

    for key in h1.history.keys():
        history[key] = (
            h1.history[key] +
            h2.history.get(key, []) 
        )

    return history


full_history = combine_histories(
    history_stage1,
    history_stage2
)
"""

# Print out the history for the CE model training
def combine_histories(h1, h2, h3):
    history = {}

    for key in h1.history.keys():
        history[key] = (
            h1.history[key] +
            h2.history.get(key, []) +
            h3.history.get(key, [])
        )

    return history


full_history = combine_histories(
    history_stage1,
    history_stage2,
    history_stage3
)

best_acc       = max(full_history['val_accuracy'])

print(f"Accuracy: {best_acc:.4f}")
print("=" * 70)
# ============================================================
# LEARNING CURVES
# ============================================================

epochs = range(
    1,
    len(full_history["accuracy"]) + 1
)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs,
    full_history["accuracy"],
    label="Train Accuracy"
)

plt.plot(
    epochs,
    full_history["val_accuracy"],
    label="Validation Accuracy"
)

plt.axvline(
    len(history_stage1.history["accuracy"]) + 0.5,
    linestyle="--",
    label="Fine-tuning begins"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("RAF-CE MobileNetV2 Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs,
    full_history["loss"],
    label="Train Loss"
)

plt.plot(
    epochs,
    full_history["val_loss"],
    label="Validation Loss"
)

plt.axvline(
    len(history_stage1.history["loss"]) + 0.5,
    linestyle="--",
    label="Fine-tuning begins"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RAF-CE MobileNetV2 Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EVALUATION
# ============================================================

test_loss, test_accuracy, test_sparse_precision = model.evaluate(
    test_ds,
    verbose=1
)

print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# ============================================================
# PREDICTIONS
# ============================================================

y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    y_true.extend(labels.numpy())
    y_pred.extend(
        np.argmax(predictions, axis=1)
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Predictions:", len(y_pred))

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

## Normalized Confusion Matrix

In [ ]:
# ============================================================
# NORMALIZED CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

# Normalize each row
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True)

# Prevent division by zero for classes with no samples
cm_normalized = np.nan_to_num(cm_normalized)

plt.figure(figsize=(13, 11))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=class_names,
    yticklabels=class_names,
    square=True
)

plt.xlabel("Predicted Emotion")
plt.ylabel("True Emotion")
plt.title("Normalized Confusion Matrix - RAF-CE MobileNetV2")

plt.xticks(rotation=60, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

# QAT to TFLite Conversion

In [ ]:
# ============================================================
# QAT → TFLITE
# ============================================================

# Save model 
model.save("rafCE_mobilenetv2_qat.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

tflite_model = converter.convert()

TFLITE_PATH = "emosys_ce.tflite"

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print(
    f"✅ Saved TFLite model: {TFLITE_PATH}"
)

print(
    f"Model size: {os.path.getsize(TFLITE_PATH) / (1024**2):.2f} MB"
)

## Sanity checks with TFLite Model

In [ ]:
# ── Load TFLite model ─────────────────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp_detail = interpreter.get_input_details()[0]
out_detail = interpreter.get_output_details()[0]

# ── Run inference on full val set ─────────────────────────────────
v_true, v_pred, v_prob = [], [], []

for images, labels in val_ds:
    for i in range(len(images)):
        img = images[i].numpy()[np.newaxis]   # (1,224,224,3)

        if inp_detail["dtype"] == np.float32:
            img = img.astype(np.float32)
        else:
            scale, zero_point = inp_detail["quantization"]
            img = (img / scale + zero_point).astype(inp_detail["dtype"])

        interpreter.set_tensor(inp_detail["index"], img)
        interpreter.invoke()

        prob = interpreter.get_tensor(out_detail["index"])[0]

        v_true.append(int(labels[i].numpy()))  # ✅ FIX: Direct integer, not argmax
        v_pred.append(np.argmax(prob))
        v_prob.append(prob)


v_true = np.array(v_true)
v_pred = np.array(v_pred)
v_prob = np.array(v_prob)

# ── Overall metrics ───────────────────────────────────────────────
overall_acc = (v_true == v_pred).mean() * 100
print(f'\n{"═"*55}')
print(f'  TFLite INT8 Evaluation — RAF-CE val set')
print(f'{"═"*55}')
print(f'  Overall Accuracy : {overall_acc:.2f}%')

# ── Per-class report ──────────────────────────────────────────────
report = classification_report(
    v_true, v_pred,
    target_names=class_names,
    digits=4,
    zero_division=0,
)

# ── Confusion matrix ──────────────────────────────────────────────
cm = confusion_matrix(v_true, v_pred, labels=range(len(class_names)))

# Check for empty classes
row_sums = cm.sum(axis=1, keepdims=True)
empty_classes = [class_names[i] for i, s in enumerate(row_sums.flatten()) if s == 0]
if empty_classes:
    print(f"⚠️  Warning: No validation samples for: {', '.join(empty_classes)}")

# Normalize safely by avoiding division by zero
cm_pct = np.divide(
    cm.astype(float) * 100,
    row_sums,
    where=row_sums != 0,
    out=np.zeros_like(cm.astype(float) * 100)
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_pct, ax=ax,
    annot=True, fmt='.1f', cmap='Blues',
    vmin=0, vmax=100,
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.4,
    annot_kws={'size': 9},
)
ax.set_title(f'TFLite INT8 — Confusion Matrix (% of true class)\n rafce val set', fontsize=11)
ax.set_xlabel('Predicted', fontsize=10)
ax.set_ylabel('True', fontsize=10)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
plt.tight_layout()

cm_path = os.path.join(f'rafce_tflite_confusion_matrix.png')
plt.savefig(cm_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\nConfusion matrix saved → {cm_path}')

In [ ]:
# ── Debug: Check what's in v_true ─────────────────────────────
unique_true = np.unique(v_true)
unique_pred = np.unique(v_pred)

print(f"\nDebug Info:")
print(f"  Unique true labels in v_true: {unique_true}")
print(f"  Unique pred labels in v_pred: {unique_pred}")
print(f"  Corresponding classes: {[class_names[i] for i in unique_true]}")
print(f"  Total samples processed: {len(v_true)}")

# Count samples per true class
for i, cls in enumerate(class_names):
    count = np.sum(v_true == i)
    print(f"    {cls}: {count}")

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

inp_detail  = interpreter.get_input_details()[0]
out_detail  = interpreter.get_output_details()[0]
dummy_input = np.random.rand(1, *IMG_SIZE, 3).astype(np.float32)

interpreter.set_tensor(inp_detail['index'], dummy_input)
interpreter.invoke()
preds = interpreter.get_tensor(out_detail['index'])

EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
print(f'Input  shape : {inp_detail["shape"]}  dtype: {inp_detail["dtype"]}')
print(f'Output shape : {out_detail["shape"]}  dtype: {out_detail["dtype"]}')
print(f'Predicted class : {class_names[np.argmax(preds)]} ({preds.max():.3f})')
print('TFLite INT8 inference OK ✓')

# Label Printing 

In [ ]:
import json

# Create a mapping of class indices to class names
label_mapping = {i: name for i, name in enumerate(class_names)}

# Save to JSON file
with open("label_CE.json", "w") as f:
    json.dump(label_mapping, f, indent=2)

print("Saved label_CE.json")